## Document Search 

In [4]:
import psycopg2
import pandas as pd
import os

def load_data(output_dir):
    os.makedirs(output_dir, exist_ok=True)
    table_names = ['erp.workflows_objects_takt', 'erp.workflows_objects_task']

    # Delete previous files in the output directory
    for filename in os.listdir(output_dir):
        file_path = os.path.join(output_dir, filename)
        if os.path.isfile(file_path):
            os.remove(file_path)

    # Database Connection and Export
    try:
        conn = psycopg2.connect(
            dbname='ost2',
            user='postgres',
            password='prodstat537(',
            host='10.187.190.17',
            port='5432'
        )

        for table in table_names:
            query = f"SELECT * FROM {table};"
            safe_filename = table.replace('.', '_') + ".json"
            out_file = os.path.join(output_dir, safe_filename)

            df = pd.read_sql(query, conn)
            df.to_json(out_file, orient='records', lines=True)  # Save as JSON

    except Exception as db_e:
        print(f"Database connection failed: {db_e}")
    finally:
        if 'conn' in locals():
            conn.close()

def main():
    # Define directories
    BASE_DIR = r"C:\Users\y7rbasav\OneDrive - Carl Zeiss AG\Desktop"
    BAD_DIR = os.path.join(BASE_DIR)

    # Load/refresh data
    load_data(BAD_DIR)

if __name__ == '__main__':
    main()

C:\Users\y7rbasav\AppData\Local\Temp\ipykernel_3444\2595781013.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [5]:
import pandas as pd
import json

def load_json_data(filepath):
    with open(filepath, "r", encoding='utf-8') as file:
        # Read multiple JSON objects (one per line)
        rows = [json.loads(line) for line in file if line.strip()]
        return {"rows": rows}

# --- Safely parse the 'attachments' column ---
def parse_attachments(val):
    if isinstance(val, str):
        try:
            return json.loads(val)
        except json.JSONDecodeError:
            return []
    return val if isinstance(val, list) else []

# --- Extract URLs and titles from attachments ---
def extract_urls_titles(attachments):
    urls, titles = [], []
    for item in attachments:
        if isinstance(item, dict):
            urls.append(item.get('url', ''))
            titles.append(item.get('title', ''))
    return urls, titles

# --- Expand attachments into dynamic columns ---
def expand_attachments(df, urls_col, titles_col):
    max_len = df[urls_col].apply(len).max()
    for i in range(max_len):
        df[f'Url-{i + 1}'] = df[urls_col].apply(lambda x: x[i] if i < len(x) else '')
        df[f'Title-{i + 1}'] = df[titles_col].apply(lambda x: x[i] if i < len(x) else '')
    return df

# --- Main processing function ---
def process_workflow_data(input_path, output_path):
    data = load_json_data(input_path)
    df = pd.DataFrame(data.get("rows", []))

    # Drop unnecessary columns
    columns_to_drop = [
        "kind", "schema", "hash", "triggers", "events",
        "actions", "confirmed_by", "for_each", "rework_upon"
    ]
    df.drop(columns=columns_to_drop, errors='ignore', inplace=True)

    # Parse and expand attachments
    df['parsed_attachments'] = df['attachments'].apply(parse_attachments)
    df['urls'], df['titles'] = zip(*df['parsed_attachments'].apply(extract_urls_titles))
    df = expand_attachments(df, 'urls', 'titles')

    # Clean up and export
    df.drop(columns=['parsed_attachments', 'urls', 'titles'], inplace=True)
    df.fillna('NA', inplace=True)
    df['Index'] = range(1, len(df) + 1)
    df.to_excel(output_path, index=False)

    # Optional: print selected columns
    print(df[['id'] + [col for col in df.columns if col.startswith('Url-') or col.startswith('Title-')]])

# --- Run the processor ---
process_workflow_data(
    input_path="erp_workflows_objects_task.json",
    output_path="processed_workflows_objects_task.xlsx"
)


          id                                              Url-1  \
0      11404  https://zeiss.sharepoint.com/:x:/r/sites/05449...   
1      11404  https://zeiss.sharepoint.com/:x:/r/sites/05449...   
2       1649                                                      
3        178  https://zeiss.sharepoint.com/:p:/r/sites/05449...   
4        170                                                      
...      ...                                                ...   
10044   1280  https://zeiss.sharepoint.com/:w:/r/sites/05449...   
10045   1279  https://zeiss.sharepoint.com/:w:/r/sites/05449...   
10046   1278  https://zeiss.sharepoint.com/:w:/r/sites/05449...   
10047   1278  https://zeiss.sharepoint.com/:w:/r/sites/05449...   
10048  10771  https://zeiss.sharepoint.com/:p:/r/sites/05449...   

                                 Title-1  \
0                           Rework Tasks   
1                           Rework Tasks   
2                                          
3        9814 - Col

In [11]:
# Load secondary workflow data for mapping
df1 = pd.read_csv('workflows_objects_takt.csv', encoding='utf-8-sig')

# Drop unnecessary columns from df1
df1.drop(columns=[
    'hash', 'kind', 'schema', 'description', 'attachments',
    'triggers', 'events', 'sap_operation', 'goods_in',
    'limitted_to', 'for_each_unit_of', 'rework_upon'
], inplace=True)

# Convert stringified sets/lists in workitem_order to Python lists
df1['workitem_order'] = df1['workitem_order'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Build mapping from workitem_order values to df1 ids
workitem_to_df1id = {}

for _, row in df1.iterrows():
    df1_id = row['id']
    for wid in row['workitem_order']:
        if wid in workitem_to_df1id:
            workitem_to_df1id[wid].append(df1_id)
        else:
            workitem_to_df1id[wid] = [df1_id]

# Map df["id"] to df1 using the mapping
df['mapped_df1_id'] = df['id'].apply(lambda x: workitem_to_df1id.get(x, [None])[0])

# Remove exact duplicate rows (keeps the first occurrence)
df = df.drop_duplicates()

df.fillna('', inplace=True)

# # Save final output
df.to_excel('cleaned_workflows_objects_task2.xlsx', index=False)

C:\Users\y7rbasav\AppData\Local\Temp\ipykernel_4600\1988738895.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)


In [12]:
df

,id,name,description,attachments,Url-1,Title-1,Url-2,Title-2,Url-3,Title-3,...,Title-13,Url-14,Title-14,Url-15,Title-15,Url-16,Title-16,Url-17,Title-17,mapped_df1_id
0,11404,Rework Tasks /Tests to be repeated,After a part change / column intervention plea...,"[{""url"": ""https://zeiss.sharepoint.com/:x:/r/s...",https://zeiss.sharepoint.com/:x:/r/sites/05449...,Rework Tasks,,,,,...,,,,,,,,,,681.0
2,1649,Final Gun Vaccum - VP,NA,[],,,,,,,...,,,,,,,,,,14.0
3,178,When pumped to spec open up manual valves and CIV,Start when system vacuum is ≤ 2.0e-5 mbar,"[{""url"": ""https://zeiss.sharepoint.com/:p:/r/s...",https://zeiss.sharepoint.com/:p:/r/sites/05449...,9814 - Column Bakeout procedure,,,,,...,,,,,,,,,,14.0
4,170,New Task,NA,[],,,,,,,...,,,,,,,,,,13.0
5,187,New Task,NA,[],,,,,,,...,,,,,,,,,,21.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9981,1167,Install Production Test Licence,This is located in Volume Manufacturing in Sha...,"[{""url"": ""https://zeiss.sharepoint.com/:w:/r/s...",https://zeiss.sharepoint.com/:w:/r/sites/05449...,SmartSEM 8.0 Installtion and Licencing,https://zeiss.sharepoint.com/:w:/r/sites/05449...,ZEN Core EM 3.11 Installation and Test,https://zeiss.sharepoint.com/:f:/r/sites/05449...,Link To SmartSEM and ZEN Production Licences,...,,,,,,,,,,403.0
9982,1291,Install Emergency Off Circuit,NA,[],,,,,,,...,,,,,,,,,,285.0
9983,6931,Stage Drift Program,Please ensure a gold on carbon sample is used ...,"[{""url"": ""https://zeiss.sharepoint.com/:w:/r/s...",https://zeiss.sharepoint.com/:w:/r/sites/05449...,Stage Drift WI,https://zeiss.sharepoint.com/sites/05449/Volum...,Stage Drift Test Program,,,...,,,,,,,,,,382.0
9985,921,Water Flow Tests,If the water temperature constantly reads 26 d...,[],,,,,,,...,,,,,,,,,,6.0


In [13]:
import pandas as pd
import json
import ast

# Load both CSV files
df_task = pd.read_csv("workflows_objects_task.csv", encoding='utf-8-sig')
df_takt = pd.read_csv("workflows_objects_takt.csv", encoding='utf-8-sig')

In [14]:
# Convert stringified sets/lists in workitem_order to Python lists
df_takt['workitem_order'] = df_takt['workitem_order'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Build a mapping from workitem_order values to df_takt ids
workitem_to_mappedid = {}
for _, row in df_takt.iterrows():
    mapped_id = row['id']
    for wid in row['workitem_order']:
        workitem_to_mappedid[wid] = mapped_id

In [15]:
exploded_rows = []
for _, row in df_task.iterrows():
    task_id = row['id']
    try:
        actions = json.loads(row['actions']) if isinstance(row['actions'], str) else []
    except Exception:
        actions = []
    for action in actions:
        if isinstance(action, dict):
            text = action.get('text', '')
            exploded_rows.append({"id": task_id, "text": text})

df_exploded = pd.DataFrame(exploded_rows)

In [16]:
df_exploded["mapped_df1_id"] = df_exploded["id"].map(workitem_to_mappedid)

In [17]:
df_exploded = df_exploded[["mapped_df1_id", "id", "text"]]

In [19]:
df_exploded.to_excel("exploded_and_mapped_actions.xlsx", index=False)
print(df_exploded)

       mapped_df1_id     id                                               text
0              681.0  11404                                        mn2 and csv
1               14.0   1649                    IGP 1 Pressure (≤ 9.0e-10 mbar)
2               14.0   1649                                     IGP 2 Pressure
3               40.0    186  Install 3DSM Software and Record Software Vers...
4               40.0    186          Set Up Image Using One segment of the BSD
...              ...    ...                                                ...
21769          285.0   3242     Installation of Zen Core EM if Hasap is fitted
21770          285.0   3242                     Set BIOS AC Power loss to ‘ON’
21771          285.0   3242     Set the Time and Date on the PC to current UK 
21772          285.0   3242         Stick PC Configuration Sheet to Side of PC
21773          285.0   3242                 Attach Cable Markers to USB Cables

[21774 rows x 3 columns]


In [1]:
import pandas as pd
import ast
import json

In [7]:
import pandas as pd

# Create DataFrame for Table 1
data1 = pd.read_excel("erp_workflows_objects_task.xlsx")

data2 = pd.read_excel("erp_workflows_objects_takt.xlsx")

# Merge the two DataFrames on 'id'
result = pd.merge(data2, data1, on='id')

# Select the desired columns: subid, id, name, tname
result = result[['sub_id', 'id', 'name', 'Takt Name']]

result.to_excel("final_copy_merged.xlsx")

# Display the result
print(result)


      sub_id    id                                        name  \
0          1     1                                Cut Pipe Kit   
1          1     2                     Assemble Quick Coupling   
2          1     3                  Assemble water flow switch   
3          1     4  Assemble Ion pump PSU to mounting brackets   
4          5    49   Check Plinth Paperwork is complete for T3   
...      ...   ...                                         ...   
6440       2  1264                               Fit Stage PCB   
6441       2  1264                Fit Stage PCB (6 Axis Stage)   
6442       2  1264                         Fit Stage PCB 6axis   
6443       2  1264                               Fit Stage PCB   
6444       2  1264                               Fit Stage PCB   

            Takt Name  
0        Pre Assembly  
1        Pre Assembly  
2        Pre Assembly  
3        Pre Assembly  
4     Takt 4 Assembly  
...               ...  
6440  Takt 1 Assembly  
6441  Takt 1 As

In [9]:
result = result.drop_duplicates(subset='name')
result.to_excel("final_copy_merged.xlsx")

In [ ]:
C:\Users\y7rbasav\AppData\Local\anaconda3\Lib\site-packages\numpy\core\_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
Traceback (most recent call last):
  File "C:\Users\y7rbasav\OneDrive - Carl Zeiss AG\Desktop\Correlation\app.py", line 114, in <module>
    preds, stig_cols = train_and_predict_stig_balance_ap_stig(excel_file, user_df)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\y7rbasav\OneDrive - Carl Zeiss AG\Desktop\Correlation\app.py", line 81, in train_and_predict_stig_balance_ap_stig
    X_test_transformed = boxcox_transformer.transform(X_test)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\y7rbasav\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\_set_output.py", line 140, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\y7rbasav\AppData\Local\anaconda3\Lib\site-packages\sklearn\preprocessing\_data.py", line 3150, in transform
    X = self._check_input(X, in_fit=False, check_positive=True, check_shape=True)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\y7rbasav\AppData\Local\anaconda3\Lib\site-packages\sklearn\preprocessing\_data.py", line 3339, in _check_input
    raise ValueError(
ValueError: The Box-Cox transformation can only be applied to strictly positive data


In [8]:
# Load the main task data
data = pd.read_csv("takts.csv", encoding='utf-8-sig')

# Drop unnecessary columns
data.drop(columns=[
    'description', 'modified', 'deleted', 'correlation_id', 'limitted_to', 
    'for_each_unit_of', 'sap_operation', 'goods_in', 'schema'], inplace=True)

In [9]:
data

,id,shop_floor,name,task_order
0,29,4,Photograph Macro (10 Mins Setup Time),{11381}
1,15,3,Takt 1 Assembly,"{186,187,188,189,190,191,192,193,194,195}"
2,3,1,Takt 2 Assembly,"{11125,11126,24,25,26,27,10936,10962,10963,28,..."
3,18,3,Takt 4 Assembly,"{218,219,220,221,222,223,224,225,226,227,228,229}"
4,402,6,Plasma Clean,{3256}
...,...,...,...,...
169,880,5,Takt TEM Prep,{}
170,879,5,Takt 11.1 UniGIS Einstellen,{}
171,875,5,Takt 11.2: Orsay GIS Einbau (GIS 2),{}
172,878,5,Takt 11: Uni GIS Einbau,{}


In [7]:
data.columns

Index(['id', 'shop_floor', 'name', 'description', 'task_order', 'modified',
       'deleted', 'correlation_id', 'limitted_to', 'for_each_unit_of',
       'sap_operation', 'goods_in', 'schema'],
      dtype='object')

In [14]:
import itertools

def cartesiam_explode(row, cols):
    # For each column remove curly braces and split into list
    lists = [row[col].strip('{}').split(',') for col in cols]
    # Get all combinations of the lists
    for combination in itertools.product(*lists):
        new_row = row.copy()
        for i, col in enumerate(cols):
            new_row[col] = combination[i]
        yield new_row

cols_to_explode = ['task_order']
rows = []
for _, row in data.iterrows():
    rows.extend(cartesiam_explode(row, cols_to_explode))

df_exploded = pd.DataFrame(rows)
df_exploded

,id,shop_floor,name,task_order
0,29,4,Photograph Macro (10 Mins Setup Time),11381
1,15,3,Takt 1 Assembly,186
1,15,3,Takt 1 Assembly,187
1,15,3,Takt 1 Assembly,188
1,15,3,Takt 1 Assembly,189
...,...,...,...,...
169,880,5,Takt TEM Prep,
170,879,5,Takt 11.1 UniGIS Einstellen,
171,875,5,Takt 11.2: Orsay GIS Einbau (GIS 2),
172,878,5,Takt 11: Uni GIS Einbau,


In [17]:
# Save final output
df_exploded.to_csv('cleaned_takts.csv', encoding='utf-8-sig', index=False)
df_exploded

,id,shop_floor,name,task_order
0,29,4,Photograph Macro (10 Mins Setup Time),11381
1,15,3,Takt 1 Assembly,186
1,15,3,Takt 1 Assembly,187
1,15,3,Takt 1 Assembly,188
1,15,3,Takt 1 Assembly,189
...,...,...,...,...
169,880,5,Takt TEM Prep,
170,879,5,Takt 11.1 UniGIS Einstellen,
171,875,5,Takt 11.2: Orsay GIS Einbau (GIS 2),
172,878,5,Takt 11: Uni GIS Einbau,


In [16]:
df_exploded

,id,shop_floor,name,task_order
0,29,4,Photograph Macro (10 Mins Setup Time),11381
1,15,3,Takt 1 Assembly,186
1,15,3,Takt 1 Assembly,187
1,15,3,Takt 1 Assembly,188
1,15,3,Takt 1 Assembly,189
...,...,...,...,...
169,880,5,Takt TEM Prep,
170,879,5,Takt 11.1 UniGIS Einstellen,
171,875,5,Takt 11.2: Orsay GIS Einbau (GIS 2),
172,878,5,Takt 11: Uni GIS Einbau,


In [19]:
data_2 = pd.read_csv('shop_floors.csv', encoding='utf-8-sig')
data_2

,id,name,description,organization,takt_order,modified,deleted,correlation_id,daily_task,template,layout,owner,commands
0,21,EVO Column,\N,2,"{522,698}",\N,\N,e3592aa9-a5d5-42ec-999e-61895fdd5efb,\N,ColumnLogbook.docx,\N,1,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."
1,3,Stage Build Logbook,NaN,2,"{15,16,17,18,19}",2019-06-16 16:40:14+00,\N,c35c891b-4744-4782-9a35-7a28566af380,\N,\N,\N,1,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."
2,0,Workflow Pool,\N,2,{},\N,\N,05922557-0c55-4dab-9614-a1603c4f0d16,\N,\N,\N,1,[]
3,8,Goods Out,\N,2,"{150,709}",\N,\N,aface803-c558-430f-a547-f4f62f582c11,\N,\N,\N,1,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."
4,6,Sigma Final Test,NaN,2,"{701,285,376,402,377,378,379,380,381,382,383,3...",\N,\N,8ced200c-ed96-42af-8a96-3c7214d98597,"[{""text"": ""EHT ="", ""type"": ""numeric"", ""unit"": ...",FinalTestLogbook.docx,final-test,1,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."
5,13,EVO Final Test,Record the system vacuum daily and the column ...,2,"{719,403,642,643,644,645,646,647,648,649,650,6...",2021-09-21 12:06:34+00,\N,a36e0203-ad53-4287-9925-b23328236571,"[{""mask"": ""X.XXe-XX"", ""text"": ""System Vacuum (...",FinalTestLogbook.docx,final-test,1,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."
6,38,Sigma Plinth,\N,2,"{655,656,670,671,717,672,674,675,677,676,678,679}",\N,\N,78f5e17c-26be-4b9c-ab3f-4e6870dba0a2,\N,NSEPlinthLogbook.docx,\N,1,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."
7,1,NSE Plinth,Assembling plinthes for GeminiSEM and Crossbeam.,2,"{2,1,3,702,4,703,720,5,6}",\N,\N,4e5c8cb4-3dd2-4a19-8704-ebfcd9bf2256,\N,NSEPlinthLogbook.docx,plinth-build,1,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."
8,2,FE Column,"Assembling Columns for Sigma, GeminiSEM, and C...",2,"{713,708,707,7,8,9,10,12,11,13,14}",\N,\N,b373df6d-4f68-4443-8441-02a0b5b6d1d3,\N,ColumnLogbook.docx,\N,38,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."
9,48,EVO Plinth,\N,2,"{658,659,696,680,660,661,662,663,664,665,669,6...",\N,\N,6b267e78-8631-4d0d-8854-4cd3b51770ee,\N,NSEPlinthLogbook.docx,\N,1,"[{""icon"": ""printer"", ""mutation"": ""printLogbook..."


In [20]:
data_2.columns

Index(['id', 'name', 'description', 'organization', 'takt_order', 'modified',
       'deleted', 'correlation_id', 'daily_task', 'template', 'layout',
       'owner', 'commands'],
      dtype='object')

In [22]:
data_2.drop(columns= ['description', 'organization', 'modified',
       'deleted', 'correlation_id', 'daily_task', 'template', 'layout',
       'owner', 'commands'], inplace=True)

data_2

,id,name,takt_order
0,21,EVO Column,"{522,698}"
1,3,Stage Build Logbook,"{15,16,17,18,19}"
2,0,Workflow Pool,{}
3,8,Goods Out,"{150,709}"
4,6,Sigma Final Test,"{701,285,376,402,377,378,379,380,381,382,383,3..."
5,13,EVO Final Test,"{719,403,642,643,644,645,646,647,648,649,650,6..."
6,38,Sigma Plinth,"{655,656,670,671,717,672,674,675,677,676,678,679}"
7,1,NSE Plinth,"{2,1,3,702,4,703,720,5,6}"
8,2,FE Column,"{713,708,707,7,8,9,10,12,11,13,14}"
9,48,EVO Plinth,"{658,659,696,680,660,661,662,663,664,665,669,6..."


In [23]:
import itertools

def cartesiam_explode_1(row, cols):
    # For each column remove curly braces and split into list
    lists = [row[col].strip('{}').split(',')for col in cols]
    # Get all combinations of the lists
    for combination in itertools.product(*lists):
        new_row = row.copy()
        for i, col in enumerate(cols):
            new_row[col] = combination[i]
        yield new_row

cols_to_explode = ['takt_order']
rows = []
for _, row in data_2.iterrows():
    rows.extend(cartesiam_explode_1(row, cols_to_explode))

df_exploded_2 = pd.DataFrame(rows)
df_exploded_2

,id,name,takt_order
0,21,EVO Column,522
0,21,EVO Column,698
1,3,Stage Build Logbook,15
1,3,Stage Build Logbook,16
1,3,Stage Build Logbook,17
...,...,...,...
11,4,NSE Final Test,105
11,4,NSE Final Test,705
11,4,NSE Final Test,45
11,4,NSE Final Test,46


In [24]:
# Save final output
df_exploded_2.to_csv('cleaned_shop_floors.csv', encoding='utf-8-sig', index=False)
df_exploded_2

,id,name,takt_order
0,21,EVO Column,522
0,21,EVO Column,698
1,3,Stage Build Logbook,15
1,3,Stage Build Logbook,16
1,3,Stage Build Logbook,17
...,...,...,...
11,4,NSE Final Test,105
11,4,NSE Final Test,705
11,4,NSE Final Test,45
11,4,NSE Final Test,46


In [25]:
data_3 = pd.read_csv('workflow_groups.csv', encoding='utf-8-sig')
data_3

,workflow,group,shop_floor
0,1,1,1
1,1,1,2
2,1,2,4
3,2,1,5
4,3,1,38
5,3,1,2
6,3,2,6
7,3,3,8
8,1,3,8
9,7,1,21


In [26]:
workflow_map = {
    1: "GeminiSEM & Crossbeam",
    2: "Crossbeam Integration",
    3: "Sigma",
    7: "EVO"
}

data_3['workflow'] = data_3['workflow'].map(workflow_map).fillna(data_3['workflow'])
data_3

,workflow,group,shop_floor
0,GeminiSEM & Crossbeam,1,1
1,GeminiSEM & Crossbeam,1,2
2,GeminiSEM & Crossbeam,2,4
3,Crossbeam Integration,1,5
4,Sigma,1,38
5,Sigma,1,2
6,Sigma,2,6
7,Sigma,3,8
8,GeminiSEM & Crossbeam,3,8
9,EVO,1,21


In [27]:
data_3.to_csv('cleaned_workflow_groups.csv', encoding='utf-8-sig', index=False)
data_3

,workflow,group,shop_floor
0,GeminiSEM & Crossbeam,1,1
1,GeminiSEM & Crossbeam,1,2
2,GeminiSEM & Crossbeam,2,4
3,Crossbeam Integration,1,5
4,Sigma,1,38
5,Sigma,1,2
6,Sigma,2,6
7,Sigma,3,8
8,GeminiSEM & Crossbeam,3,8
9,EVO,1,21


In [2]:
import pandas as pd

In [4]:
df_exploded_3 = pd.read_csv('workflows_objects_sub_assembly_workflow.csv', encoding='utf-8-sig')


import itertools

def cartesiam_explode_1(row, cols):
    # For each column remove curly braces and split into list
    lists = [row[col].strip('{}').split(',')for col in cols]
    # Get all combinations of the lists
    for combination in itertools.product(*lists):
        new_row = row.copy()
        for i, col in enumerate(cols):
            new_row[col] = combination[i]
        yield new_row

cols_to_explode = ['workitem_order']
rows = []
for _, row in df_exploded_3.iterrows():
    rows.extend(cartesiam_explode_1(row, cols_to_explode))

df_exploded_3 = pd.DataFrame(rows)
df_exploded_3 = df_exploded_3.drop_duplicates()

df_exploded_3.to_csv('cleaned_workflows_objects_sub_assembly_workflow.csv', encoding='utf-8-sig', index=False)

df_exploded_3

,hash,kind,schema,id,name,description,attachments,triggers,events,workitem_order,sap_operation
0,\xafafe46d838fccc7ac962048ebf554fc506520ebb5d6...,logbook,sub-assembly-workflow,29,Differential Pressure Stage,\N,[],\N,[],,\N
1,\x246fb46dbcdf2f7964d21e3aeacb884110a54d82c3a4...,logbook,sub-assembly-workflow,29,New Log Book,\N,[],\N,[],,\N
2,\x065c6bfccbeab37482857c891267b22bb8eb5d9e09a5...,logbook,sub-assembly-workflow,28,New Log Book,\N,[],\N,[],,\N
3,\x39230786f96f931cae88e37d821d4c725e1893321f87...,logbook,sub-assembly-workflow,25,Sigma Objective 80° VP,Sigma Objective 80° Supermarket logbook,[],\N,"[{""event"": ""completed"", ""actions"": [{""bookYiel...",599,\N
3,\x39230786f96f931cae88e37d821d4c725e1893321f87...,logbook,sub-assembly-workflow,25,Sigma Objective 80° VP,Sigma Objective 80° Supermarket logbook,[],\N,"[{""event"": ""completed"", ""actions"": [{""bookYiel...",600,\N
...,...,...,...,...,...,...,...,...,...,...,...
283,\x6efef7a439d585f01c20cebd82f5a66deb23ede0c590...,logbook,sub-assembly-workflow,3,Sigma Single Condenser 349520-9017-000,\N,[],\N,[],,\N
284,\xf21f61e03663e632543e7d315f8372c0b2f8f4e53828...,logbook,sub-assembly-workflow,3,Sigma Single Condenser 349520-9017-000,80 Degree Objective Lens Supermarket,[],\N,"[{""event"": ""completed"", ""actions"": [{""bookYiel...",529,\N
284,\xf21f61e03663e632543e7d315f8372c0b2f8f4e53828...,logbook,sub-assembly-workflow,3,Sigma Single Condenser 349520-9017-000,80 Degree Objective Lens Supermarket,[],\N,"[{""event"": ""completed"", ""actions"": [{""bookYiel...",530,\N
284,\xf21f61e03663e632543e7d315f8372c0b2f8f4e53828...,logbook,sub-assembly-workflow,3,Sigma Single Condenser 349520-9017-000,80 Degree Objective Lens Supermarket,[],\N,"[{""event"": ""completed"", ""actions"": [{""bookYiel...",531,\N


In [ ]:
def process_directory(bad_data_dir, good_data_dir, logger=None):
    os.makedirs(good_data_dir, exist_ok=True)

    for filename in os.listdir(bad_data_dir):
        if filename.endswith('.json'):
            base = os.path.splitext(filename)[0]
            input_path = os.path.join(bad_data_dir, filename)

            if base in CLEANING_FUNCTIONS:
                # Allow for one or more functions per base
                funcs = CLEANING_FUNCTIONS[base]
                if not isinstance(funcs, list):  # Support both single and multiple functions
                    funcs = [funcs]

                # Reset counter for each file/base
                for idx, func in enumerate(funcs, start=1):
                    output_path = os.path.join(good_data_dir, f"{base}_{idx}.xlsx")

                    if logger:
                        logger.info(f"Processing {filename} (output #{idx}) with {func.__name__}")
                    else:
                        print(f"Processing {filename} (output #{idx}) with {func.__name__}")

                    # Call the cleaning function
                    func(input_path, output_path)
            else:
                msg = f"No cleaning function for {filename}, skipping."
                if logger:
                    logger.warning(msg)
                else:
                    print(msg)

In [ ]:
def process_workflow_objects_tasks_combined(input_path, attachment_output, actions_output):
    # --- Load line-delimited JSON objects from input file once ---
    with open(input_path, "r", encoding='utf-8') as file:
        rows = [json.loads(line) for line in file if line.strip()]
    df_src = pd.DataFrame(rows)

    # --- Process Attachments ---
    df_attachments = df_src.copy()
    columns_to_drop_attachments = [
        "kind", "schema", "hash", "triggers", "events",
        "actions", "confirmed_by", "for_each", "rework_upon"
    ]
    df_attachments = df_attachments.drop(
        columns=[col for col in columns_to_drop_attachments if col in df_attachments.columns],
        errors='ignore'
    )

    if 'description' in df_attachments.columns:
        df_attachments = df_attachments[df_attachments['description'].astype(str).str.strip().str.endswith('~')]


    def parse_attachments(val):
        if isinstance(val, str):
            try:
                return json.loads(val)
            except Exception:
                return []
        return val if isinstance(val, list) else []

    if 'attachments' in df_attachments.columns:
        df_attachments['parsed_attachments'] = df_attachments['attachments'].apply(parse_attachments)

        def extract_urls_titles(attachments):
            urls, titles = [], []
            for item in attachments:
                if isinstance(item, dict):
                    urls.append(item.get('url', ''))
                    titles.append(item.get('title', ''))
            return urls, titles

        df_attachments['urls'], df_attachments['titles'] = zip(*df_attachments['parsed_attachments'].apply(extract_urls_titles))

        max_len = df_attachments['urls'].apply(len).max() if len(df_attachments['urls']) > 0 else 0
        for i in range(max_len):
            df_attachments[f'Url-{i+1}'] = df_attachments['urls'].apply(lambda x: x[i] if i < len(x) else '')
            df_attachments[f'Title-{i+1}'] = df_attachments['titles'].apply(lambda x: x[i] if i < len(x) else '')

        df_attachments.drop(columns=['parsed_attachments', 'urls', 'titles'], inplace=True)

        # Deduplicate on only hashable columns
        url_cols = [f'Url-{i+1}' for i in range(max_len)]
        title_cols = [f'Title-{i+1}' for i in range(max_len)]
        subset_cols = ['name', 'id'] + url_cols + title_cols
        subset_cols = [c for c in subset_cols if c in df_attachments.columns]
        df_attachments.drop_duplicates(subset=subset_cols, inplace=True)

        df_attachments.fillna('NA', inplace=True)
        df_attachments['Index'] = range(1, len(df_attachments) + 1)
        df_attachments.to_excel(attachment_output, index=False)
    else:
        pd.DataFrame({'error': ['no attachments column found']}).to_excel(attachment_output, index=False)

    # --- Process Actions ---
    df_actions = df_src.copy()
    columns_to_drop_actions = [
        "kind", "schema", "hash", "triggers", "events",
        "attachments", "confirmed_by", "for_each", "rework_upon"
    ]
    df_actions = df_actions.drop(
        columns=[col for col in columns_to_drop_actions if col in df_actions.columns],
        errors='ignore'
    )

    def parse_actions(val):
        if isinstance(val, str):
            try:
                return json.loads(val)
            except Exception:
                return []
        elif isinstance(val, list):
            return val
        return []

    if 'actions' in df_actions.columns:
        df_actions['parsed_actions'] = df_actions['actions'].apply(parse_actions)
        rows_expanded = []
        for _, row in df_actions.iterrows():
            id_, name = row.get('id'), row.get('name')
            actions = row['parsed_actions']
            if not isinstance(actions, list):
                continue
            for action in actions:
                if isinstance(action, dict) and 'text' in action:
                    rows_expanded.append({'id': id_, 'name': name, 'actions': action['text']})
        df_actions_out = pd.DataFrame(rows_expanded, columns=['id', 'name', 'actions'])
        df_actions_out['id'] = df_actions_out['id'].fillna('NA')
        df_actions_out['name'] = df_actions_out['name'].fillna('')
        df_actions_out['actions'] = df_actions_out['actions'].fillna('')
        df_actions_out['Index'] = range(1, len(df_actions_out) + 1)
        df_actions_out.to_excel(actions_output, index=False)
    else:
        pd.DataFrame({'error': ['no actions column found']}).to_excel(actions_output, index=False)